# Week 3 - Advanced Data Analysis and Visualization in Logistics

This notebook simulates a hypothetical logistics dataset, performs exploratory data analysis (EDA), creates visualizations, and derives logistics insights and recommendations.

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## 2. Simulate the Logistics Dataset

In [ ]:
rng = np.random.default_rng(42)
n = 500

regions = rng.choice(["North", "South", "East", "West"], n, p=[.25,.25,.23,.27])
transport = rng.choice(["Road", "Rail", "Air"], n, p=[.62,.23,.15])
priority = rng.choice(["Standard", "Express"], n, p=[.72,.28])
distance = np.clip(rng.gamma(2.6, 170, n) + 50, 60, 1400)
volume = np.clip(rng.lognormal(3.0, .55, n), 5, 150)
weight = np.clip(volume * rng.uniform(.65, 1.9, n), 5, 240)

base_days = {"Road":3.8, "Rail":5.2, "Air":1.5}
delivery = (
    np.array([base_days[t] for t in transport])
    + distance / np.array([260 if t=="Road" else 190 if t=="Rail" else 700 for t in transport])
    + rng.normal(0, .75, n)
    + np.where(priority=="Express", -.8, 0)
    + np.where(regions=="East", .35, 0)
)
delivery = np.clip(delivery, .5, None)

cost_per_km = {"Road":2.9, "Rail":1.75, "Air":8.4}
cost = (
    distance * np.array([cost_per_km[t] for t in transport])
    + volume * rng.uniform(18, 38, n)
    + weight * rng.uniform(2.0, 5.5, n)
    + np.where(priority=="Express", 650, 0)
    + rng.normal(0, 380, n)
)
cost = np.clip(cost, 500, None)

delay_prob = np.clip(
    .08 + .00055*distance + .035*(delivery>7) + .05*(regions=="East") + .025*(transport=="Rail"),
    .03, .75
)
delayed = rng.random(n) < delay_prob

df = pd.DataFrame({
    "Region": regions,
    "Transport_Mode": transport,
    "Priority": priority,
    "Distance_km": distance,
    "Shipment_Volume_units": volume,
    "Weight_kg": weight,
    "Delivery_Time_days": delivery,
    "Transport_Cost_INR": cost,
    "Delayed": np.where(delayed, "Yes", "No")
})

df.head()

## 3. Basic Dataset Inspection

In [ ]:
print('Dataset shape:', df.shape)
display(df.head(10))
display(df.describe(include='all'))

## 4. Central Tendencies and Key Logistics Metrics

In [ ]:
metrics = pd.Series({
    "Average delivery time (days)": df["Delivery_Time_days"].mean(),
    "Median delivery time (days)": df["Delivery_Time_days"].median(),
    "Average transport cost (INR)": df["Transport_Cost_INR"].mean(),
    "Median transport cost (INR)": df["Transport_Cost_INR"].median(),
    "Average distance (km)": df["Distance_km"].mean(),
    "Overall delay rate (%)": (df["Delayed"] == "Yes").mean() * 100
})
display(metrics.to_frame('Value'))

## 5. Distribution of Delivery Time

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(df["Delivery_Time_days"], bins=25, edgecolor="black")
plt.title("Distribution of Delivery Time")
plt.xlabel("Delivery Time (days)")
plt.ylabel("Number of Shipments")
plt.tight_layout()
plt.show()

**Interpretation:** The histogram shows the typical delivery duration and highlights the spread and long-tail shipments that take considerably longer than normal.

## 6. Transport Cost by Mode

In [ ]:
plt.figure(figsize=(8,5))
groups = [df.loc[df.Transport_Mode==m, "Transport_Cost_INR"] for m in ["Road","Rail","Air"]]
plt.boxplot(groups, tick_labels=["Road","Rail","Air"], showfliers=False)
plt.title("Transport Cost by Mode")
plt.xlabel("Transport Mode")
plt.ylabel("Transport Cost (INR)")
plt.tight_layout()
plt.show()

**Interpretation:** Air transport has the strongest cost premium, while rail generally provides a lower-cost option for suitable shipments.

## 7. Distance vs Transport Cost

In [ ]:
plt.figure(figsize=(8,5))
for mode in ["Road","Rail","Air"]:
    d = df[df["Transport_Mode"] == mode]
    plt.scatter(d["Distance_km"], d["Transport_Cost_INR"], label=mode, alpha=.55)
plt.xlabel("Distance (km)")
plt.ylabel("Transport Cost (INR)")
plt.title("Distance vs Transport Cost")
plt.legend()
plt.tight_layout()
plt.show()

**Interpretation:** Transportation cost generally increases as shipment distance increases. The different slopes reflect the distinct cost structures of road, rail, and air transport.

## 8. Transport Mode Performance

In [ ]:
mode_summary = df.groupby("Transport_Mode").agg(
    Shipments=("Transport_Mode","count"),
    Avg_Delivery_Days=("Delivery_Time_days","mean"),
    Avg_Cost_INR=("Transport_Cost_INR","mean"),
    Delay_Rate=("Delayed", lambda x: (x=="Yes").mean()*100)
).sort_values("Avg_Cost_INR", ascending=False)
display(mode_summary)

plot_df = mode_summary.loc[["Road","Rail","Air"]]
x = np.arange(len(plot_df))
plt.figure(figsize=(8,5))
plt.bar(x - .18, plot_df["Avg_Delivery_Days"], width=.36, label="Avg delivery days")
plt.bar(x + .18, plot_df["Delay_Rate"], width=.36, label="Delay rate (%)")
plt.xticks(x, plot_df.index)
plt.title("Transport Mode Performance")
plt.ylabel("Metric value")
plt.legend()
plt.tight_layout()
plt.show()

**Interpretation:** Mode selection involves a trade-off among speed, cost, and reliability. Air is best suited to urgent shipments, while lower-cost modes can be used when service requirements allow.

## 9. Correlation Analysis

In [ ]:
numeric_cols = ["Distance_km","Shipment_Volume_units","Weight_kg","Delivery_Time_days","Transport_Cost_INR"]
corr = df[numeric_cols].corr()
display(corr)

plt.figure(figsize=(8,6))
im = plt.imshow(corr.values, aspect="auto")
plt.colorbar(im, label="Correlation")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
plt.yticks(range(len(corr.index)), corr.index)
plt.title("Correlation Matrix of Key Numeric Variables")
for i in range(corr.shape[0]):
    for j in range(corr.shape[1]):
        plt.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center", fontsize=8)
plt.tight_layout()
plt.show()

**Interpretation:** Correlation helps identify variables that move together. Distance is an important factor related to both delivery time and transportation cost, although correlation alone does not prove causation.

## 10. Delay Rate by Region

In [ ]:
region_summary = df.groupby("Region").agg(
    Shipments=("Region","count"),
    Avg_Delivery_Days=("Delivery_Time_days","mean"),
    Avg_Cost_INR=("Transport_Cost_INR","mean"),
    Delay_Rate=("Delayed", lambda x: (x=="Yes").mean()*100)
).sort_values("Delay_Rate", ascending=False)
display(region_summary)

rp = region_summary.loc[["North","South","East","West"]]
plt.figure(figsize=(8,5))
plt.bar(rp.index, rp["Delay_Rate"])
plt.title("Delay Rate by Region")
plt.xlabel("Region")
plt.ylabel("Delayed Shipments (%)")
plt.tight_layout()
plt.show()

**Interpretation:** Regional differences can reveal potential bottlenecks related to routes, carrier performance, congestion, handling, or scheduling.

## 11. Analytical Insights and Recommendations

- Air transport is the fastest option but carries a significant cost premium.
- Rail can provide a lower-cost alternative for suitable long-distance shipments.
- Distance is a major driver of transportation cost and delivery time.
- Regions with higher delay rates should receive focused operational investigation.
- Transport mode should be selected using both cost and service requirements.
- Delivery-time percentiles should be monitored alongside averages to detect long-tail delays.
- Real historical data should be used in future analysis with additional variables such as carrier, origin, destination, fuel prices, weather, and warehouse processing time.